# Adult Census Income — 파이프라인 단계별 탐색 노트북

`main.py` 의 파이프라인을 구성하는 `src` 모듈들을 그대로 불러와, 각 단계(1~6)를 셀 단위로 실행하면서
중간 결과(데이터, 통계, 그래프, 모델 성능)를 직접 확인합니다.

**파이프라인 개요**

1. 데이터 수집/로딩 (`src.load`)
2. 전처리 (`src.clean`)
3. 통계 분석 (`src.stats`)
4. 시각화 (`src.visualize`)
5. ML 파이프라인 (`src.model`)
6. 리포트 생성 (`src.report`)


## 0. 환경 설정

프로젝트 루트를 `sys.path` 에 추가하고, `src` 모듈과 경로 상수를 불러옵니다.

In [1]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src import clean, load, model, report, stats, visualize

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUTS_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR


PosixPath('/Users/minchae/workspace/skala-gwangju-4class-team2')

## 1. 데이터 수집 및 로딩 (`src.load`)

UCI adult.data 를 다운로드(이미 있으면 재사용)한 뒤, Pandas와 Polars 양쪽으로 로딩해 성능·타입 차이를 비교합니다.

In [2]:
raw_path = load.fetch_raw_data(RAW_DIR)
raw_path


PosixPath('/Users/minchae/workspace/skala-gwangju-4class-team2/data/raw/adult.data')

In [3]:
pdf, pd_time = load.load_with_pandas(raw_path)
pldf, pl_time = load.load_with_polars(raw_path)
pdf.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
comparison = load.compare_pandas_polars(pdf, pldf, pd_time, pl_time)
comparison


전처리 전 기본 EDA: 데이터 shape과 컬럼별 결측치 개수를 확인합니다.

In [ ]:
print("shape:", pdf.shape)
na_counts = pdf.isna().sum()
na_counts[na_counts > 0]


## 2. 전처리 (`src.clean`)

결측치 제거(dropna) → 중복 제거 → 문자열 strip → target(income) 0/1 이진화를 수행하고,
전처리 결과를 `data/processed/adult_clean.csv` 로 저장합니다.

In [ ]:
df, clean_summary = clean.clean_data(pdf)
clean_summary


In [ ]:
df.to_csv(PROCESSED_DIR / "adult_clean.csv", index=False)
df.head()


## 3. 통계 분석 (`src.stats`)

기술통계·상관분석과 함께 가설 H1(근무시간), H2(교육수준), H4(혼인/관계)를 검정합니다.
H3(capital_gain)은 로지스틱 회귀 계수가 필요하므로 5단계(ML) 이후에 검정합니다.

In [ ]:
describe_df = stats.descriptive_stats(df)
describe_df


In [ ]:
corr_df = stats.correlation_analysis(df)
corr_df


In [ ]:
h1 = stats.test_h1_hours_worked(df)
h1


In [ ]:
h2 = stats.test_h2_education(df)
h2


In [ ]:
h4 = stats.test_h4_marital_relationship(df)
h4


## 4. 시각화 (`src.visualize`)

Seaborn으로 정적 차트(boxplot, correlation heatmap)를, Plotly로 인터랙티브 막대 차트(education, marital_status별 고소득 비율)를 생성합니다.

In [ ]:
from IPython.display import Image, IFrame, display

boxplot_path = visualize.plot_income_hours_boxplot(df, FIGURES_DIR / "income_hours_boxplot.png")
display(Image(filename=str(boxplot_path)))


In [ ]:
heatmap_path = visualize.plot_correlation_heatmap(df, FIGURES_DIR / "correlation_heatmap.png")
display(Image(filename=str(heatmap_path)))


In [ ]:
edu_html_path = visualize.plot_income_ratio_by_education_interactive(
    df, FIGURES_DIR / "income_ratio_by_education.html"
)
display(IFrame(src=str(edu_html_path), width="100%", height=500))


In [ ]:
marital_html_path = visualize.plot_income_ratio_by_marital_status_interactive(
    df, FIGURES_DIR / "income_ratio_by_marital_status.html"
)
display(IFrame(src=str(marital_html_path), width="100%", height=500))


## 5. ML 파이프라인 (`src.model`)

train/test 분할 → 로지스틱 회귀 학습 → 평가지표 산출 → 상위 계수 추출 → 모델 저장을 수행합니다.

In [ ]:
result = model.train_and_evaluate(df)
result["metrics"]


In [ ]:
print(result["classification_report"])


In [ ]:
result["confusion_matrix"]


In [ ]:
result["coef_df"]


H3(capital_gain) 검정: 상관계수와 로지스틱 회귀 계수를 함께 확인합니다.

In [ ]:
capital_gain_coef = model.get_capital_gain_coefficient(result["coef_df"])
h3 = stats.test_h3_capital_gain(df, model_coef=capital_gain_coef)
h3


In [ ]:
model_path = model.save_model(result["pipeline"], OUTPUTS_DIR / "model.joblib")
model_path


## 6. 리포트 생성 (`src.report`)

지금까지의 모든 결과를 종합해 `outputs/report.md` 를 자동으로 작성합니다.

In [ ]:
context = {
    "pandas_polars_comparison": comparison,
    "clean_summary": clean_summary,
    "describe_md": report.df_to_markdown(describe_df, index_label="statistic"),
    "corr_md": report.df_to_markdown(corr_df, index_label="feature"),
    "h1": h1,
    "h2": h2,
    "h3": h3,
    "h4": h4,
    "metrics": result["metrics"],
    "confusion_matrix": result["confusion_matrix"],
    "classification_report": result["classification_report"],
    "coef_md": report.df_to_markdown(result["coef_df"], show_index=False),
    "n_train": result["n_train"],
    "n_test": result["n_test"],
    "artifacts": {
        "전처리 완료 데이터": PROCESSED_DIR / "adult_clean.csv",
        "학습된 모델": model_path,
        "Seaborn: income vs hours_per_week boxplot": boxplot_path,
        "Seaborn: 상관관계 heatmap": heatmap_path,
        "Plotly: education별 고소득 비율": edu_html_path,
        "Plotly: marital_status별 고소득 비율": marital_html_path,
    },
}
report_path = report.generate_report(context, OUTPUTS_DIR / "report.md")
report_path


생성된 리포트 미리보기:

In [ ]:
from IPython.display import Markdown

Markdown(report_path.read_text(encoding="utf-8"))
